<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/04_Foundations/mathematics/05_statistics_hypothesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Statistics & Hypothesis Testing

Statistical inference lets us draw conclusions about populations from samples. This is essential for A/B testing, model evaluation, and data-driven decision making.

**Topics:**
- Central Limit Theorem (simulation)
- Confidence intervals
- Hypothesis testing framework
- t-test, chi-squared test, ANOVA
- p-values, Type I/II errors, statistical power
- Multiple testing correction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

## 1. Central Limit Theorem — Simulation

**CLT:** The distribution of sample means approaches Normal as sample size increases, regardless of the underlying distribution.

In [ ]:
n_experiments = 5000
sample_sizes = [1, 5, 30, 100]

# Use a very non-normal distribution: Exponential(lambda=1)
# True mean = 1, True std = 1

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, n in enumerate(sample_sizes):
    # Draw many samples of size n, compute each sample mean
    sample_means = [np.random.exponential(1, n).mean() for _ in range(n_experiments)]
    sample_means = np.array(sample_means)
    
    # Top: distribution of individual samples (n=1) or population shape
    ax_top = axes[0, idx]
    if n == 1:
        x_exp = np.linspace(0, 6, 200)
        ax_top.plot(x_exp, stats.expon.pdf(x_exp), 'r-', lw=2)
        ax_top.fill_between(x_exp, stats.expon.pdf(x_exp), alpha=0.3, color='red')
        ax_top.set_title('Population\n(Exponential)')
    else:
        ax_top.hist(sample_means, bins=50, density=True, color='steelblue', alpha=0.7)
        x_norm = np.linspace(sample_means.min(), sample_means.max(), 200)
        mu_expected = 1.0  # true mean
        sigma_expected = 1.0 / np.sqrt(n)  # CLT: sigma/sqrt(n)
        ax_top.plot(x_norm, stats.norm.pdf(x_norm, mu_expected, sigma_expected), 
                    'r-', lw=2, label='N(μ,σ/√n)')
        ax_top.legend(fontsize=8)
        ax_top.set_title(f'Sample Means\n(n={n})')
    ax_top.set_xlabel('Value'); ax_top.set_ylabel('Density')
    ax_top.grid(True, alpha=0.3)
    
    # Bottom: QQ plot to assess normality
    ax_bot = axes[1, idx]
    if n > 1:
        stats.probplot(sample_means, dist='norm', plot=ax_bot)
        ax_bot.set_title(f'Q-Q Plot (n={n})')
        ax_bot.grid(True, alpha=0.3)
    else:
        ax_bot.text(0.5, 0.5, 'N/A\n(single obs.)', ha='center', va='center',
                    transform=ax_bot.transAxes, fontsize=12)
        ax_bot.axis('off')

plt.suptitle('Central Limit Theorem: Exponential → Normal as n increases', 
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Confidence Intervals

In [ ]:
# True population: Normal(mu=50, sigma=10)
true_mu = 50
true_sigma = 10
n = 30  # sample size
alpha = 0.05  # 95% CI

# Simulate 100 experiments; how many CIs capture true mean?
n_experiments = 100
captured = 0

fig, ax = plt.subplots(figsize=(12, 8))
ax.axvline(true_mu, color='k', lw=2, label=f'True μ = {true_mu}', zorder=5)

for i in range(n_experiments):
    sample = np.random.normal(true_mu, true_sigma, n)
    sample_mean = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(n)  # standard error
    
    # 95% CI using t-distribution (appropriate for small n)
    t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
    margin = t_crit * se
    ci_low, ci_high = sample_mean - margin, sample_mean + margin
    
    contains_true = ci_low <= true_mu <= ci_high
    if contains_true:
        captured += 1
    
    color = 'steelblue' if contains_true else 'red'
    ax.plot([ci_low, ci_high], [i, i], color=color, lw=1.5, alpha=0.7)
    ax.scatter(sample_mean, i, color=color, s=20, zorder=3)

ax.set_xlabel('Value'); ax.set_ylabel('Experiment #')
ax.set_title(f'95% Confidence Intervals: {captured}/100 capture true μ\n(expect ~95)')
ax.legend(); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
print(f'Coverage rate: {captured}%  (expected: ~95%)')

## 3. Hypothesis Testing Framework

1. **H₀ (null hypothesis):** Status quo — no effect, no difference
2. **H₁ (alternative):** What you're trying to show
3. **Test statistic:** Measure of how far data is from H₀
4. **p-value:** P(seeing data this extreme | H₀ is true)
5. **Decision:** Reject H₀ if p-value < α (significance level)

In [ ]:
# Two-sample t-test: do two groups have different means?
# Example: conversion rates for control vs treatment
np.random.seed(42)

control = np.random.normal(loc=100, scale=15, size=50)    # control group revenue
treatment = np.random.normal(loc=108, scale=15, size=50)  # treatment group revenue

# Compute manually
x1, x2 = control.mean(), treatment.mean()
s1, s2 = control.std(ddof=1), treatment.std(ddof=1)
n1, n2 = len(control), len(treatment)

# Pooled standard error
se = np.sqrt(s1**2/n1 + s2**2/n2)
t_stat = (x2 - x1) / se

# Degrees of freedom (Welch-Satterthwaite)
df = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
p_val_manual = 2 * stats.t.sf(abs(t_stat), df)  # two-tailed

# Using scipy
t_stat_scipy, p_val_scipy = stats.ttest_ind(control, treatment)

print('Two-Sample t-Test')
print(f'Control mean:   {x1:.2f}')
print(f'Treatment mean: {x2:.2f}')
print(f'Difference:     {x2-x1:.2f}')
print(f'T-statistic:    {t_stat:.4f}')
print(f'p-value:        {p_val_scipy:.4f}')
print(f'\nDecision: ', end='')
if p_val_scipy < 0.05:
    print(f'REJECT H₀ (p={p_val_scipy:.4f} < 0.05) → Significant difference!')
else:
    print(f'FAIL to reject H₀ (p={p_val_scipy:.4f} > 0.05) → No significant difference')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(control, bins=15, alpha=0.6, color='steelblue', label=f'Control (μ={x1:.1f})', density=True)
ax.hist(treatment, bins=15, alpha=0.6, color='darkorange', label=f'Treatment (μ={x2:.1f})', density=True)
ax.set_title('Distribution Comparison'); ax.set_xlabel('Revenue')
ax.legend(); ax.grid(True, alpha=0.3)

# t-distribution and rejection region
ax2 = axes[1]
x_t = np.linspace(-5, 5, 300)
t_dist = stats.t.pdf(x_t, df=df)
ax2.plot(x_t, t_dist, 'k-', lw=2, label='t-distribution')
t_crit = stats.t.ppf(0.975, df=df)
# Shade rejection regions
ax2.fill_between(x_t, t_dist, where=(x_t > t_crit), color='red', alpha=0.5, label=f'Rejection region (α=0.05)')
ax2.fill_between(x_t, t_dist, where=(x_t < -t_crit), color='red', alpha=0.5)
ax2.axvline(t_stat, color='green', ls='--', lw=2, label=f'Observed t={t_stat:.2f}')
ax2.set_title('t-Test: Test Statistic vs Rejection Region')
ax2.set_xlabel('t'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Type I & II Errors and Statistical Power

In [ ]:
# Type I error (α): False positive — reject H₀ when it's true
# Type II error (β): False negative — fail to reject H₀ when it's false
# Power = 1 - β: probability of detecting a real effect

from scipy.stats import norm

alpha = 0.05
effect_size = 0.5   # Cohen's d: standardized mean difference
n_range = np.arange(10, 300, 5)

# Power for two-sample t-test (approximation using z)
def compute_power(n, effect_size, alpha=0.05):
    """Approximate power for two-sample t-test."""
    se = np.sqrt(2/n)
    z_alpha = norm.ppf(1 - alpha/2)
    z_power = effect_size / se - z_alpha
    return norm.cdf(z_power)

powers = [compute_power(n, effect_size) for n in n_range]

# Sample size for 80% power
n_80 = n_range[np.argmax(np.array(powers) >= 0.80)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(n_range, powers, 'b-', lw=2)
ax.axhline(0.80, color='red', ls='--', label='80% power threshold')
ax.axhline(0.90, color='green', ls='--', label='90% power threshold')
ax.axvline(n_80, color='red', ls=':', label=f'n={n_80} for 80% power')
ax.set_xlabel('Sample size per group'); ax.set_ylabel('Power')
ax.set_title(f'Statistical Power vs Sample Size\n(effect size d={effect_size}, α={alpha})')
ax.legend(); ax.grid(True, alpha=0.3)

# Error types visualization
ax2 = axes[1]
x = np.linspace(-4, 8, 400)
h0 = norm.pdf(x, 0, 1)    # H₀: no effect
h1 = norm.pdf(x, 2.5, 1)  # H₁: effect exists
z_crit = norm.ppf(1 - 0.05)

ax2.plot(x, h0, 'b-', lw=2, label='H₀ (no effect)')
ax2.plot(x, h1, 'r-', lw=2, label='H₁ (effect exists)')
ax2.fill_between(x, h0, where=(x > z_crit), color='blue', alpha=0.3, label=f'α: Type I error')
ax2.fill_between(x, h1, where=(x < z_crit), color='red', alpha=0.3, label='β: Type II error')
ax2.fill_between(x, h1, where=(x > z_crit), color='green', alpha=0.3, label='Power = 1-β')
ax2.axvline(z_crit, color='black', ls='--', lw=2, label=f'Critical value z={z_crit:.2f}')
ax2.set_title('Type I Error (α), Type II Error (β), and Power')
ax2.set_xlabel('Test statistic'); ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Need n≥{n_80} per group for 80% power (effect d={effect_size}, α={alpha})')

## 5. Multiple Testing Correction

In [ ]:
# Problem: if you test 100 hypotheses at α=0.05,
# you expect ~5 false positives by chance!

np.random.seed(42)
n_tests = 100
alpha = 0.05

# Simulate p-values from null hypothesis (no real effects)
p_values = np.random.uniform(0, 1, n_tests)

# Uncorrected: how many "significant" at alpha=0.05?
n_significant_uncorrected = (p_values < alpha).sum()

# Bonferroni correction: divide alpha by number of tests
alpha_bonferroni = alpha / n_tests
n_significant_bonferroni = (p_values < alpha_bonferroni).sum()

# Benjamini-Hochberg (FDR control) — less conservative
sorted_p = np.sort(p_values)
k = np.arange(1, n_tests + 1)
bh_threshold = k * alpha / n_tests
bh_significant = (sorted_p < bh_threshold).sum()

print(f'Multiple Testing Correction ({n_tests} tests, α={alpha})')
print(f'Expected false positives (all null): {n_tests * alpha:.0f}')
print(f'\nUncorrected: {n_significant_uncorrected} "significant" results')
print(f'Bonferroni (α/m={alpha_bonferroni:.4f}): {n_significant_bonferroni} significant')
print(f'Benjamini-Hochberg (FDR≤5%): {bh_significant} significant')

plt.figure(figsize=(10, 5))
plt.scatter(range(n_tests), sorted_p, s=20, alpha=0.7, label='p-values (sorted)')
plt.axhline(alpha, color='red', ls='--', label=f'Uncorrected α={alpha}')
plt.axhline(alpha_bonferroni, color='blue', ls='--', label=f'Bonferroni α/m={alpha_bonferroni:.4f}')
plt.plot(range(n_tests), bh_threshold, 'g-', lw=2, label='BH threshold (FDR=5%)')
plt.xlabel('Rank (sorted p-value)'); plt.ylabel('p-value')
plt.title('Multiple Testing Correction Methods')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

| Test | Use case | scipy function |
|------|---------|---------------|
| One-sample t-test | Is sample mean different from known value? | `stats.ttest_1samp` |
| Two-sample t-test | Do two groups have different means? | `stats.ttest_ind` |
| Paired t-test | Before/after on same subjects | `stats.ttest_rel` |
| Chi-squared test | Independence of categorical variables | `stats.chi2_contingency` |
| ANOVA | Differences across 3+ groups | `stats.f_oneway` |
| Mann-Whitney U | Non-parametric two-group test | `stats.mannwhitneyu` |

**Next:** [06_bayesian_thinking.ipynb](06_bayesian_thinking.ipynb)